# Lab 1: Table Partitioning in PostgreSQL
## ISM 6562 — Big Data for Business Applications | Week 04

---

### The Scenario

Remember the IoT sensor database from Week 03? You learned to use `EXPLAIN ANALYZE` and indexes to speed up queries on 100,000 sensor readings. That was a great start.

But your campus IoT deployment has been running for over a year now. The `sensor_readings` table has grown to **500,000 rows** spanning January 2025 through February 2026. Queries that used to be fast are slowing down, and the operations team is asking: *"Can we do better — on a single server — before we start talking about distributing data across multiple machines?"*

The answer is **table partitioning** — splitting one large table into smaller, more manageable pieces that PostgreSQL can query selectively.

### What You'll Learn

1. How to establish a **baseline** with `EXPLAIN ANALYZE` on a non-partitioned table
2. **Range partitioning** — split data by time periods (monthly)
3. **List partitioning** — split data by category (sensor unit type)
4. **Hash partitioning** — distribute data evenly across buckets
5. How partitioning affects **indexes** and enables **instant data removal**

### Prerequisites

Start the Docker environment:

```bash
cd partitioning-lab
docker compose up -d
```

Wait about 30–60 seconds for the database to initialize with 500K rows.

## 1. Setup and Connection

In [ ]:
import psycopg2
import time

# Connection parameters
CONN_PARAMS = {
    'host': 'localhost',
    'port': 5502,
    'dbname': 'sensor_db',
    'user': 'student',
    'password': 'student'
}

conn = psycopg2.connect(**CONN_PARAMS)
conn.autocommit = True
cur = conn.cursor()

print("Connected to sensor_db on port 5502")

In [ ]:
# Verify the data loaded correctly
cur.execute("SELECT COUNT(*) FROM sensor_readings")
total = cur.fetchone()[0]

cur.execute("SELECT MIN(reading_time), MAX(reading_time) FROM sensor_readings")
min_ts, max_ts = cur.fetchone()

cur.execute("SELECT COUNT(DISTINCT sensor_id) FROM sensor_readings")
sensor_count = cur.fetchone()[0]

print(f"Total readings:  {total:,}")
print(f"Sensors:         {sensor_count}")
print(f"Date range:      {min_ts.strftime('%Y-%m-%d')} to {max_ts.strftime('%Y-%m-%d')}")
print(f"\nThat's {total // sensor_count:,} readings per sensor — about {total // sensor_count // 24} days of hourly data.")

## 2. Baseline: Querying the Non-Partitioned Table

Before we partition anything, let's establish a baseline. We'll use `EXPLAIN ANALYZE` (just like in Week 03) to see how PostgreSQL handles queries on the full 500K-row table.

In [ ]:
def explain_analyze(cursor, query, params=None):
    """Run EXPLAIN ANALYZE and print the query plan."""
    cursor.execute(f"EXPLAIN ANALYZE {query}", params)
    plan = cursor.fetchall()
    for row in plan:
        print(row[0])
    return plan

In [ ]:
# Baseline: Query readings for a single month
print("BASELINE: All readings for March 2025 (non-partitioned)")
print("=" * 60)

explain_analyze(cur, """
    SELECT COUNT(*), AVG(value)
    FROM sensor_readings
    WHERE reading_time >= '2025-03-01'
      AND reading_time <  '2025-04-01'
""")

print("\n** Notice: PostgreSQL must scan ALL 500K rows to find March data.")
print("   There's no way to skip irrelevant months — it's all one big table.")

In [ ]:
# How big is this table on disk?
cur.execute("""
    SELECT pg_size_pretty(pg_total_relation_size('sensor_readings')) AS total_size,
           pg_size_pretty(pg_relation_size('sensor_readings'))       AS table_size
""")
total_size, table_size = cur.fetchone()
print(f"Table size (data only):  {table_size}")
print(f"Total size (+ indexes):  {total_size}")

## 3. Range Partitioning (by Month)

Range partitioning is the most common strategy for **time-series data**. We'll split `sensor_readings` into monthly partitions. When you query for a specific month, PostgreSQL only scans that one partition — this is called **partition pruning**.

### Step 1: Create the partitioned table

We need a new table definition with `PARTITION BY RANGE`. Then we create one partition per month.

In [ ]:
# Create the range-partitioned table
cur.execute("DROP TABLE IF EXISTS readings_by_month CASCADE")

cur.execute("""
    CREATE TABLE readings_by_month (
        reading_id    INTEGER        NOT NULL,
        sensor_id     INTEGER        NOT NULL,
        reading_time  TIMESTAMP      NOT NULL,
        value         NUMERIC(10,2)  NOT NULL,
        unit          VARCHAR(10)    NOT NULL
    ) PARTITION BY RANGE (reading_time)
""")

print("Created partitioned table: readings_by_month")
print("Partition key: reading_time (RANGE)")

In [ ]:
# Create monthly partitions from Jan 2025 through Mar 2026
months = []
year, month = 2025, 1
while (year, month) <= (2026, 3):
    next_month = month + 1
    next_year = year
    if next_month > 12:
        next_month = 1
        next_year = year + 1
    
    partition_name = f"readings_{year}_{month:02d}"
    start_date = f"{year}-{month:02d}-01"
    end_date = f"{next_year}-{next_month:02d}-01"
    
    cur.execute(f"""
        CREATE TABLE {partition_name} PARTITION OF readings_by_month
        FOR VALUES FROM ('{start_date}') TO ('{end_date}')
    """)
    months.append(partition_name)
    
    month = next_month
    year = next_year

print(f"Created {len(months)} monthly partitions:")
for m in months:
    print(f"  {m}")

In [ ]:
# Copy data from the original table into the partitioned table
start = time.time()

cur.execute("""
    INSERT INTO readings_by_month (reading_id, sensor_id, reading_time, value, unit)
    SELECT reading_id, sensor_id, reading_time, value, unit
    FROM sensor_readings
""")

elapsed = time.time() - start

cur.execute("SELECT COUNT(*) FROM readings_by_month")
count = cur.fetchone()[0]
print(f"Copied {count:,} rows into partitioned table in {elapsed:.1f}s")

In [ ]:
# Check the distribution of rows across partitions
print("Rows per partition:")
print("=" * 40)
for partition in months:
    cur.execute(f"SELECT COUNT(*) FROM {partition}")
    count = cur.fetchone()[0]
    if count > 0:
        bar = '#' * (count // 1000)
        print(f"  {partition:20s}  {count:>7,}  {bar}")

### Step 2: Partition Pruning in Action

Now let's run the same query — March 2025 readings — but against the partitioned table. Watch for partition pruning in the `EXPLAIN ANALYZE` output.

In [ ]:
print("PARTITIONED: All readings for March 2025")
print("=" * 60)

explain_analyze(cur, """
    SELECT COUNT(*), AVG(value)
    FROM readings_by_month
    WHERE reading_time >= '2025-03-01'
      AND reading_time <  '2025-04-01'
""")

print("\n** Look for 'readings_2025_03' in the plan — PostgreSQL only scans that partition!")
print("   All other partitions are pruned (skipped) automatically.")

In [ ]:
# Compare: query spanning two months
print("PARTITIONED: Readings for March-April 2025 (spans 2 partitions)")
print("=" * 60)

explain_analyze(cur, """
    SELECT COUNT(*), AVG(value)
    FROM readings_by_month
    WHERE reading_time >= '2025-03-01'
      AND reading_time <  '2025-05-01'
""")

print("\n** Now 2 partitions are scanned, but the rest are still pruned.")

### Step 3: The Primary Key Constraint

Here's something that surprises many people: in a partitioned table, the **primary key must include the partition key**. Let's see why.

In [ ]:
# This will FAIL — PK on reading_id alone doesn't include partition key
print("Attempting: PRIMARY KEY (reading_id) on partitioned table...")
print("=" * 60)

try:
    cur.execute("""
        ALTER TABLE readings_by_month
        ADD CONSTRAINT pk_readings_month PRIMARY KEY (reading_id)
    """)
except psycopg2.Error as e:
    print(f"ERROR: {e.pgerror.strip()}")
    conn.rollback()

print("\nWhy? Each partition enforces uniqueness independently.")
print("Without the partition key, PostgreSQL can't guarantee global uniqueness.")

In [ ]:
# This WORKS — include reading_time in the PK
print("Attempting: PRIMARY KEY (reading_id, reading_time) — includes partition key...")
print("=" * 60)

cur.execute("""
    ALTER TABLE readings_by_month
    ADD CONSTRAINT pk_readings_month PRIMARY KEY (reading_id, reading_time)
""")

print("Success! PK must include the partition key column.")
print("\nThis is a trade-off: partitioning gives you pruning,")
print("but constrains how you can define uniqueness.")

## 4. List Partitioning (by Sensor Unit)

Range partitioning works great for time-series. But what if you want to partition by **category**? For example, temperature, humidity, and pressure readings have different units (`F`, `%`, `hPa`) and different query patterns.

List partitioning lets you assign specific values to specific partitions.

In [ ]:
# Create list-partitioned table by unit type
cur.execute("DROP TABLE IF EXISTS readings_by_unit CASCADE")

cur.execute("""
    CREATE TABLE readings_by_unit (
        reading_id    INTEGER        NOT NULL,
        sensor_id     INTEGER        NOT NULL,
        reading_time  TIMESTAMP      NOT NULL,
        value         NUMERIC(10,2)  NOT NULL,
        unit          VARCHAR(10)    NOT NULL
    ) PARTITION BY LIST (unit)
""")

# Create one partition per unit type
cur.execute("""
    CREATE TABLE readings_temperature PARTITION OF readings_by_unit
    FOR VALUES IN ('F')
""")

cur.execute("""
    CREATE TABLE readings_humidity PARTITION OF readings_by_unit
    FOR VALUES IN ('%')
""")

cur.execute("""
    CREATE TABLE readings_pressure PARTITION OF readings_by_unit
    FOR VALUES IN ('hPa')
""")

print("Created list-partitioned table with 3 partitions:")
print("  readings_temperature  (unit = 'F')")
print("  readings_humidity     (unit = '%')")
print("  readings_pressure     (unit = 'hPa')")

In [ ]:
# Load data
cur.execute("""
    INSERT INTO readings_by_unit (reading_id, sensor_id, reading_time, value, unit)
    SELECT reading_id, sensor_id, reading_time, value, unit
    FROM sensor_readings
""")

# Check distribution
print("Rows per partition:")
print("=" * 50)
for name, unit_val in [('readings_temperature', 'F'), ('readings_humidity', '%'), ('readings_pressure', 'hPa')]:
    cur.execute(f"SELECT COUNT(*) FROM {name}")
    count = cur.fetchone()[0]
    pct = count / 500000 * 100
    print(f"  {name:25s}  {count:>7,}  ({pct:.0f}%)")

print("\nDistribution matches our sensor mix: 20 temp + 20 humidity + 10 pressure")

In [ ]:
# Demonstrate list partition pruning
print("LIST PARTITION PRUNING: Query only pressure readings")
print("=" * 60)

explain_analyze(cur, """
    SELECT COUNT(*), AVG(value)
    FROM readings_by_unit
    WHERE unit = 'hPa'
""")

print("\n** Only the readings_pressure partition is scanned.")
print("   Temperature and humidity partitions are completely skipped.")

## 5. Hash Partitioning (by Sensor ID)

What if your data doesn't have natural categories or time boundaries? **Hash partitioning** distributes rows evenly across a fixed number of buckets using a hash function. This is useful when you want even distribution but don't have a logical split.

We'll hash by `sensor_id` to spread the load across 4 partitions.

In [ ]:
# Create hash-partitioned table
cur.execute("DROP TABLE IF EXISTS readings_by_hash CASCADE")

cur.execute("""
    CREATE TABLE readings_by_hash (
        reading_id    INTEGER        NOT NULL,
        sensor_id     INTEGER        NOT NULL,
        reading_time  TIMESTAMP      NOT NULL,
        value         NUMERIC(10,2)  NOT NULL,
        unit          VARCHAR(10)    NOT NULL
    ) PARTITION BY HASH (sensor_id)
""")

# Create 4 hash partitions
for i in range(4):
    cur.execute(f"""
        CREATE TABLE readings_hash_{i} PARTITION OF readings_by_hash
        FOR VALUES WITH (MODULUS 4, REMAINDER {i})
    """)

print("Created hash-partitioned table with 4 buckets")
print("Partition key: sensor_id (HASH, modulus 4)")

In [ ]:
# Load data
cur.execute("""
    INSERT INTO readings_by_hash (reading_id, sensor_id, reading_time, value, unit)
    SELECT reading_id, sensor_id, reading_time, value, unit
    FROM sensor_readings
""")

# Check distribution — should be roughly even
print("Hash partition distribution:")
print("=" * 50)
for i in range(4):
    cur.execute(f"SELECT COUNT(*) FROM readings_hash_{i}")
    count = cur.fetchone()[0]
    bar = '#' * (count // 5000)
    print(f"  readings_hash_{i}  {count:>7,}  {bar}")

print("\nHash partitioning gives roughly even distribution,")
print("but you can't predict which sensor lands in which bucket.")

In [ ]:
# Hash pruning: query for a specific sensor_id
print("HASH PARTITION PRUNING: Query sensor_id = 7")
print("=" * 60)

explain_analyze(cur, """
    SELECT COUNT(*), AVG(value)
    FROM readings_by_hash
    WHERE sensor_id = 7
""")

print("\n** PostgreSQL hashes sensor_id=7 and goes directly to the right bucket.")
print("   But range queries (sensor_id BETWEEN 5 AND 15) can't be pruned — ")
print("   the hash function doesn't preserve ordering.")

In [ ]:
# Demonstrate: range queries CAN'T be pruned with hash partitioning
print("HASH PARTITION: Range query (sensor_id BETWEEN 5 AND 15)")
print("=" * 60)

explain_analyze(cur, """
    SELECT COUNT(*), AVG(value)
    FROM readings_by_hash
    WHERE sensor_id BETWEEN 5 AND 15
""")

print("\n** All 4 partitions are scanned — hash partitioning can't prune range queries.")
print("   Use range partitioning if your queries involve ranges.")

## 6. Per-Partition Indexes

When you create an index on a partitioned table, PostgreSQL creates a **separate index on each partition**. This means each index is smaller and more cache-friendly than one giant index.

In [ ]:
# Create an index on the range-partitioned table
cur.execute("""
    CREATE INDEX idx_readings_month_sensor
    ON readings_by_month (sensor_id, reading_time)
""")

# Also create the same index on the original non-partitioned table for comparison
cur.execute("""
    CREATE INDEX idx_readings_orig_sensor
    ON sensor_readings (sensor_id, reading_time)
""")

print("Created indexes on both tables. Let's compare sizes.")

In [ ]:
# Compare index sizes: partitioned vs non-partitioned
print("INDEX SIZE COMPARISON")
print("=" * 60)

# Non-partitioned: single large index
cur.execute("""
    SELECT pg_size_pretty(pg_relation_size('idx_readings_orig_sensor'))
""")
orig_size = cur.fetchone()[0]
print(f"Non-partitioned index:  {orig_size} (one big index)")

# Partitioned: many small indexes
print(f"\nPartitioned indexes (one per partition):")
total_part_size = 0
for partition in months:
    cur.execute(f"""
        SELECT indexname, pg_relation_size(indexname::regclass)
        FROM pg_indexes
        WHERE tablename = '{partition}'
          AND indexdef LIKE '%sensor_id%reading_time%'
    """)
    rows = cur.fetchall()
    for idx_name, idx_bytes in rows:
        total_part_size += idx_bytes
        cur.execute(f"SELECT pg_size_pretty({idx_bytes})")
        pretty = cur.fetchone()[0]
        print(f"  {partition:20s}  {pretty}")

cur.execute(f"SELECT pg_size_pretty({total_part_size})")
total_pretty = cur.fetchone()[0]
print(f"\nTotal partitioned index size: {total_pretty}")
print(f"\nEach partition's index is small enough to fit in memory,")
print(f"making lookups within a partition very fast.")

## 7. Instant Data Removal

One of partitioning's biggest operational advantages: **dropping a partition is nearly instant**, while `DELETE` must scan and remove rows one by one. This matters when you need to archive or purge old data.

In [ ]:
# Time a DELETE on the non-partitioned table
# (We'll delete Jan 2025 data and then re-insert it)
cur.execute("""
    SELECT COUNT(*) FROM sensor_readings
    WHERE reading_time >= '2025-01-01' AND reading_time < '2025-02-01'
""")
jan_count = cur.fetchone()[0]
print(f"January 2025 readings: {jan_count:,}")

start = time.time()
cur.execute("""
    DELETE FROM sensor_readings
    WHERE reading_time >= '2025-01-01' AND reading_time < '2025-02-01'
""")
delete_time = time.time() - start
print(f"\nDELETE {jan_count:,} rows: {delete_time:.3f}s")

# Re-insert the data so we don't lose it
cur.execute("""
    INSERT INTO sensor_readings (reading_id, sensor_id, reading_time, value, unit)
    SELECT reading_id, sensor_id, reading_time, value, unit
    FROM readings_by_month
    WHERE reading_time >= '2025-01-01' AND reading_time < '2025-02-01'
""")
print("(Re-inserted data for comparison)")

In [ ]:
# Time a DROP PARTITION (detach + drop)
start = time.time()
cur.execute("""
    ALTER TABLE readings_by_month
    DETACH PARTITION readings_2025_01
""")
cur.execute("DROP TABLE readings_2025_01")
drop_time = time.time() - start

print(f"DROP partition:  {drop_time:.3f}s")
print(f"DELETE rows:     {delete_time:.3f}s")
print(f"\nDrop is ~{delete_time/drop_time:.0f}x faster!")
print("\nDropping a partition is a metadata operation — no row-by-row scanning.")
print("This is why time-series databases almost always use partitioning.")

In [ ]:
# Re-create the January partition for completeness
cur.execute("""
    CREATE TABLE readings_2025_01 PARTITION OF readings_by_month
    FOR VALUES FROM ('2025-01-01') TO ('2025-02-01')
""")
cur.execute("""
    INSERT INTO readings_by_month (reading_id, sensor_id, reading_time, value, unit)
    SELECT reading_id, sensor_id, reading_time, value, unit
    FROM sensor_readings
    WHERE reading_time >= '2025-01-01' AND reading_time < '2025-02-01'
""")
print("Re-created January 2025 partition.")

## 8. Summary: Partitioning Strategies Compared

| Strategy | Partition Key | Best For | Pruning Works With | Limitation |
|----------|--------------|----------|-------------------|------------|
| **Range** | `reading_time` | Time-series, date ranges | Range queries (`BETWEEN`, `>=`, `<`) | Uneven partition sizes if data is seasonal |
| **List** | `unit` | Categories, regions, types | Equality queries (`=`, `IN`) | Must list every possible value |
| **Hash** | `sensor_id` | Even distribution, point lookups | Equality queries (`=`) only | No range pruning, can't predict placement |

### Key Takeaways

1. **Partition pruning** is the main performance benefit — queries only scan relevant partitions
2. **Primary keys must include the partition key** — a trade-off for partition independence
3. **Per-partition indexes** are smaller and more cache-friendly
4. **DROP partition is instant** — critical for data lifecycle management
5. All of this happens on a **single server** — no network overhead, no distributed coordination

## 9. Exercises

Try these on your own:

### Exercise 1: Pruning Investigation
Run `EXPLAIN ANALYZE` on the range-partitioned table with these queries. Which ones trigger pruning?

```sql
-- Query A: specific date range
SELECT * FROM readings_by_month WHERE reading_time = '2025-06-15 12:00:00';

-- Query B: filter on value (not the partition key)
SELECT * FROM readings_by_month WHERE value > 80;

-- Query C: filter on sensor_id (not the partition key)
SELECT * FROM readings_by_month WHERE sensor_id = 10;
```

### Exercise 2: Add a New Partition
Add an April 2026 partition to `readings_by_month`. What happens if you try to insert a reading dated April 15, 2026 *before* creating the partition?

### Exercise 3: Default Partition
Create a `DEFAULT` partition to catch rows that don't match any existing range. Try inserting a reading from 2024. Where does it go?

In [ ]:
# Space for your exercise work



## 10. Bridge to Lab 2

Partitioning is powerful — but it all happens on **one server**. What happens when:

- That server runs out of disk space?
- You need more CPU for concurrent queries?
- You need geographic locality (data near users)?

That's when you need **sharding** — distributing data across multiple independent database servers. In Lab 2, you'll build a manually-sharded PostgreSQL cluster and experience the trade-offs firsthand.

## Cleanup

When you're done, shut down the Docker environment:

```bash
cd partitioning-lab
docker compose down -v
```

In [ ]:
# Close the database connection
cur.close()
conn.close()
print("Connection closed.")